# 01 · Cohort statistics: model-free tables and figures

Produces every **data-derived** (model-free) table and figure from the clean monthly
cohorts. Nothing here depends on a fitted model, so this notebook can be run before, after
or independently of the modelling (`02_modelling.ipynb` → `03_model_results.ipynb`).

**The executed notebook is the report**: tables and figures are rendered inline and nothing
is written to disk. Set `SAVE_PAPER_ASSETS = True` in `CONFIG` to additionally write the
publication assets (`.tex` tables, 300-dpi PNG + SVG + PDF figures, cohort sizes) to
`results/paper/`.

| Output | Function | Asset stem (when saved) |
| :--- | :--- | :--- |
| Demographics by latitude band (one table per metric) | `paper_tables.demographics_table` | `rhr_demographics_thinned.tex`, `hrv_demographics_original.tex` |
| Demographic distributions (age, gender, latitude, country, panels A–F) | `pf.generate_figure_1` | `figure_1` |
| Observed seasonal 2×2 maps per metric | `pf.build_country_gender_validation` + `pf.plot_seasonal_maps_2x2` | `{metric}_validation_country_gender.csv`, `{metric}_actual_all_seasonal_map_2x2` |
| Within-user sensitivity (months per user) | `pf.make_within_user_sensitivity` | `within_user_sensitivity_combined` |
| Thinning validation (three figures) | `pf.generate_figure_a1/a2/a3` | `figure_a1..a3` |
| Cohort sizes | inline | `cohort_summary.json` |

The RHR cohort is thinned with exactly the same routine as the modelling notebook
(`sl.apply_density_based_thinning`), so the tables and figures describe the fitted cohort.
File access goes through `file_io` (`./data` or `$WPM_DATA_DIR`; see the README).

The demographic and thinning-validation figures apply minimum-count thresholds per age and
latitude band; the map figures need the reference files described in `data/README.md` in
`data/extra_data/`.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import gc
import json
import os
import sys
import time

from IPython.display import HTML, Markdown, display
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(
    0, os.path.abspath('../src')
)  # library modules (public release)
import cohort_io as cio
import file_io as fio
import paper_tables
import publication_figures as pf
import seasonality_lib as sl

In [ ]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
THIN_RHR = True  # same density-based thinning as 02_modelling.ipynb
USER_FRAC = 1.0  # fraction of users to keep after loading (1.0 = everyone; <1 for quick runs)
RANDOM_STATE = 42
RHR_COL, HRV_COL = 'avg_resting_heart_rate_bpm', 'avg_hrv'
RHR_RANGE, HRV_RANGE = (30.0, 150.0), (
    5.0,
    250.0,
)  # plausibility windows (user summaries)
# Also write the publication assets (.tex, png/svg/pdf, cohort_summary.json) to
# results/paper/? Otherwise this notebook is the report.
SAVE_PAPER_ASSETS = False

data_dir = fio.get_data_dir()
extra_dir = fio.get_extra_dir(data_dir)
paper_dir = fio.get_paper_dir(data_dir) if SAVE_PAPER_ASSETS else None
if paper_dir:
  fio.make_dirs(paper_dir)
pf.setup_nature_style()
print(
    f'data: {data_dir}\npublication assets:'
    f' {paper_dir or "not saved (SAVE_PAPER_ASSETS=False)"}'
)


def show_table(label, latex):
  """Renders a generated LaTeX table inline (preview + collapsible source)."""
  preview = paper_tables.table_preview(latex)
  display(Markdown(f'**`{label}`** — {preview.attrs.get("caption", "")}'))
  display(HTML(preview.to_html(index=False)))
  if preview.attrs.get('notes'):
    display(Markdown(f'<small>{preview.attrs["notes"]}</small>'))
  display(
      HTML(
          '<details><summary>LaTeX source</summary><pre style="font-size:80%">'
          + latex.replace('&', '&amp;').replace('<', '&lt;')
          + '</pre></details>'
      )
  )

In [ ]:
# ----------------------------------------------------------------------------
# Load the clean cohorts
# ----------------------------------------------------------------------------
COLS = [
    'user_id',
    'activity_period',
    'gender',
    'age',
    'lat_e7',
    'lng_e7',
    'country_2_alpha_cd',
    'dl_monthly',
    'avg_temp_mean',
]


def load_clean(metric, value_col):
  """Loads one cohort through cohort_io (headers / units mapped to the code's column names)."""
  t0 = time.time()
  path = cio.resolve_input_path(data_dir, metric)
  df = cio.load_cohort(data_dir, metric, columns=COLS + [value_col], path=path)
  if USER_FRAC < 1.0:
    keep = pd.Series(df['user_id'].unique()).sample(
        frac=USER_FRAC, random_state=RANDOM_STATE
    )
    df = df[df['user_id'].isin(keep)].reset_index(drop=True)
  print(
      f'{metric.upper()}: {len(df):,} user-months / {df["user_id"].nunique():,}'
      f' users ({time.time() - t0:.0f}s)'
  )
  return df


df_rhr_raw = load_clean('rhr', RHR_COL)
df_hrv = load_clean('hrv', HRV_COL)
# Calendar year of the cohort (used in table captions).
COHORT_YEAR = int(
    pd.to_datetime(df_rhr_raw['activity_period']).dt.year.mode()[0]
)

In [ ]:
# ----------------------------------------------------------------------------
# RHR thinning of the age-22 artefact - identical to the modelling notebook
# ----------------------------------------------------------------------------
if THIN_RHR:
  df_rhr, thin_stats = sl.apply_density_based_thinning(
      df_rhr_raw,
      target_col=RHR_COL,
      val_min=RHR_RANGE[0],
      val_max=RHR_RANGE[1],
      random_state=RANDOM_STATE,
  )
  rhr_summary = thin_stats['user_summary_after']
else:
  df_rhr = df_rhr_raw
  rhr_summary = sl.get_user_summary(
      df_rhr[df_rhr[RHR_COL].between(*RHR_RANGE)], RHR_COL
  )
hrv_summary = sl.get_user_summary(
    df_hrv[df_hrv[HRV_COL].between(*HRV_RANGE)], HRV_COL
)

# User-level registries (primary latitude band, mean value, age, sex) shared by
# Shared by the demographic and thinning-validation figures: built once per cohort.
t0 = time.time()
rhr_registry = pf.build_user_registry(df_rhr, val_col=RHR_COL)
hrv_registry = pf.build_user_registry(df_hrv, val_col=HRV_COL)
raw_registry = (
    pf.build_user_registry(df_rhr_raw, val_col=RHR_COL) if THIN_RHR else None
)
print(f'User registries built in {time.time() - t0:.0f}s')

## Participant demographics by latitude band

In [ ]:
show_table(
    'tab:rhr_demographics_thinned',
    paper_tables.demographics_table(
        rhr_summary, 'rhr', year=COHORT_YEAR, out_dir=paper_dir
    ),
)
show_table(
    'tab:hrv_demographics_original',
    paper_tables.demographics_table(
        hrv_summary, 'hrv', year=COHORT_YEAR, out_dir=paper_dir
    ),
)

# Cohort sizes (and the thinning effect), for the record.
cohort_summary = {
    'rhr': {
        'users': int(df_rhr['user_id'].nunique()),
        'user_months': int(len(df_rhr)),
        'thinned': bool(THIN_RHR),
        'users_before_thinning': int(df_rhr_raw['user_id'].nunique()),
        'user_months_before_thinning': int(len(df_rhr_raw)),
    },
    'hrv': {
        'users': int(df_hrv['user_id'].nunique()),
        'user_months': int(len(df_hrv)),
    },
    'user_frac': USER_FRAC,
    'generated': time.strftime('%Y-%m-%d %H:%M:%S'),
}
display(Markdown('**Cohort sizes**'))
display(pd.DataFrame({m: cohort_summary[m] for m in ('rhr', 'hrv')}).fillna(''))
if paper_dir:
  with fio.open_file(os.path.join(paper_dir, 'cohort_summary.json'), 'w') as f:
    json.dump(cohort_summary, f, indent=2)

## Demographic distributions (panels A–F)

In [ ]:
fig_1, png_1, svg_1 = pf.generate_figure_1(
    df_rhr=df_rhr,
    df_hrv=df_hrv,
    output_dir=paper_dir,
    file_prefix='figure_1',
    rhr_col=RHR_COL,
    hrv_col=HRV_COL,
    rhr_registry=rhr_registry,
    hrv_registry=hrv_registry,
)
plt.show()
if png_1:
  print(f'Saved {png_1}, {svg_1}')
plt.close('all')

## Observed seasonal maps

Observed seasonal 2×2 choropleths per metric. They only need country × sex × month means
of the observed values, which are built here from the cohorts in memory (no model involved;
saved as `{metric}_validation_country_gender.csv` next to the figures when assets are saved).

In [ ]:
# The country x sex x month means never need reference tables; the maps themselves need
# extra_data/ (Natural Earth geojson and the UNSD country table, see data/README.md).
validation = {
    metric: pf.build_country_gender_validation(
        data_dir, metric, out_dir=paper_dir, with_predictions=False, df=df
    )
    for metric, df in (('rhr', df_rhr), ('hrv', df_hrv))
}
if fio.file_exists(os.path.join(extra_dir, pf.WORLD_GEOJSON_FILE)):
  for metric in ('rhr', 'hrv'):
    pf.plot_seasonal_maps_2x2(
        data_dir,
        extra_dir,
        paper_dir,
        metrics=(metric,),
        validation_frames=validation,
    )
  plt.show()
  plt.close('all')
else:
  print(
      f'{pf.WORLD_GEOJSON_FILE} not found in {extra_dir}: map figures skipped'
  )

## Within-user sensitivity

Within-user month-to-month variability of RHR and HRV against the change in day length /
temperature (no model involved; the function reloads the needed columns itself).

In [ ]:
pf.make_within_user_sensitivity(data_dir, paper_dir)
plt.show()
plt.close('all')

## Thinning validation (before vs after)

In [ ]:
if THIN_RHR:
  for fn in (
      pf.generate_figure_a1,
      pf.generate_figure_a2,
      pf.generate_figure_a3,
  ):
    _, png, svg = fn(
        df_raw=df_rhr_raw,
        df_thinned=df_rhr,
        output_dir=paper_dir,
        rhr_col=RHR_COL,
        raw_registry=raw_registry,
        thinned_registry=rhr_registry,
    )
    plt.show()
    if png:
      print(f'Saved {png}, {svg}')
  plt.close('all')
else:
  print('THIN_RHR is False: Figures A.1-A.3 need the thinned cohort.')
del df_rhr_raw, raw_registry
gc.collect()

**Next:** fit the models with [`02_modelling.ipynb`](02_modelling.ipynb) (once per metric),
then regenerate the model-derived tables and figures with
[`03_model_results.ipynb`](03_model_results.ipynb).